# 🚀 Hunyuan3D v4 ULTRA PRO — Free SaaS Backend

> يعمل على GPU مجاني عبر Google Colab  
> يشتغل كـ API Backend مباشر + Cloudflare Tunnel للنشر العلني  
> يدعم Queue System + جيل 3D + تحميل GLB + Rate Limiting + Cache  

---

## ⚡ خلية 1: تثبيت المتطلبات

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install -q fastapi uvicorn nest_asyncio requests python-multipart aiofiles
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate scipy pillow tqdm trimesh
# تثبيت cloudflared (مش pip package — binary)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

print("✅ All dependencies installed!")

## 🧠 خلية 2: المحرك الأساسي + Queue + Rate Limiter + Cache

In [ ]:
# ============================================================
# CELL 2: Core Engine + Queue + Rate Limiter + Cache + API
# ============================================================

import os, uuid, json, time, struct
from pathlib import Path
from datetime import datetime, timedelta
from threading import Thread, Lock
from queue import Queue
from typing import Optional, Dict, Any

# ---- FastAPI ----
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import FileResponse, JSONResponse, HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# ============================================================
# CONFIG
# ============================================================
OUTPUT_DIR   = "/content/outputs"
CACHE_DIR    = "/content/cache"
THUMB_DIR    = "/content/thumbnails"
CACHE_TTL    = 3600       # 1 hour cache for files

for d in [OUTPUT_DIR, CACHE_DIR, THUMB_DIR]:
    os.makedirs(d, exist_ok=True)



# ============================================================
# FILE CACHE (Simple in-memory + disk)
# ============================================================
file_cache: Dict[str, tuple] = {}  # file_id -> (bytes, content_type, expiry_time)
cache_lock = Lock()

def get_cached_file(file_id: str) -> Optional[tuple]:
    """Get cached file bytes if still fresh."""
    with cache_lock:
        entry = file_cache.get(file_id)
        if entry:
            data, ct, expiry = entry
            if time.time() < expiry:
                return (data, ct)
            else:
                del file_cache[file_id]
    return None

def set_cached_file(file_id: str, data: bytes, content_type: str):
    """Store file in memory cache."""
    with cache_lock:
        file_cache[file_id] = (data, content_type, time.time() + CACHE_TTL)
        # Keep cache from growing too large (max ~100 entries)
        if len(file_cache) > 100:
            oldest = min(file_cache.keys(), key=lambda k: file_cache[k][2])
            del file_cache[oldest]


# ============================================================
# MODEL PLACEHOLDER (استبدل بـ الموديل الحقيقي)
# ============================================================
MODEL = None
MODEL_LOCK = Lock()

def load_hunyuan_model():
    """تحميل موديل Hunyuan3D-2 في الذاكرة."""
    global MODEL
    with MODEL_LOCK:
        if MODEL is not None:
            return MODEL

        print("🚀 Loading Hunyuan3D-2 v4 engine...")

        # ========================================
        # 🔌 REAL MODEL HOOK — استبدل السطور التالية:
        # ========================================
        # from diffusers import Hunyuan3DPipeline
        # import torch
        # MODEL = Hunyuan3DPipeline.from_pretrained(
        #     "tencent/Hunyuan3D-2",
        #     torch_dtype=torch.float16,
        #     variant="fp16"
        # )
        # MODEL.enable_model_cpu_offload()
        # MODEL.enable_vae_slicing()
        # ========================================

        # 🧪 PLACEHOLDER:
        MODEL = {"name": "Hunyuan3D-2-v4", "loaded": True, "device": "cuda"}

        print(f"✅ Model loaded: {MODEL}")
        return MODEL


# ============================================================
# SMART PROMPT ENGINE
# ============================================================
def enhance_prompt(raw_prompt: str) -> str:
    """تحسين الـ prompt بذكاء للحصول على أفضل نتيجة 3D."""
    base = raw_prompt.strip()

    enhancers = [
        "ultra detailed",
        "high quality 3D mesh",
        "PBR textures",
        "realistic materials",
        "clean topology",
        "game-ready asset",
        "centered origin",
        "proper scale"
    ]

    existing = set(base.lower().split(", "))
    unique_enhancers = [e for e in enhancers if e.lower() not in existing]

    return base + ", " + ", ".join(unique_enhancers)


# ============================================================
# GENERATION ENGINE
# ============================================================
def generate_3d_asset(prompt: str, job_id: str) -> Dict[str, Any]:
    """وظيفة التوليد الرئيسية."""
    model = load_hunyuan_model()
    enhanced = enhance_prompt(prompt)

    print(f"🎨 Generating: {enhanced[:100]}...")

    glb_path = os.path.join(OUTPUT_DIR, f"{job_id}.glb")
    png_path = os.path.join(THUMB_DIR, f"{job_id}.png")

    # ========================================
    # 🔌 REAL INFERENCE HOOK:
    # mesh = MODEL(
    #     prompt=enhanced,
    #     num_inference_steps=40,
    #     guidance_scale=7.5,
    #     output_type="mesh",
    #     generator=torch.Generator("cuda").manual_seed(42)
    # ).mesh[0]
    # mesh.export(glb_path, file_type="glb")
    # ========================================

    # 🧪 PLACEHOLDER:
    _write_valid_glb(glb_path)
    _write_thumbnail(png_path)

    file_size = os.path.getsize(glb_path)

    # Pre-cache the generated file
    with open(glb_path, 'rb') as f:
        set_cached_file(job_id, f.read(), "model/gltf-binary")
    with open(png_path, 'rb') as f:
        set_cached_file(f"thumb_{job_id}", f.read(), "image/png")

    return {
        "job_id": job_id,
        "glb_path": glb_path,
        "thumb_path": png_path,
        "prompt_original": prompt,
        "prompt_enhanced": enhanced,
        "file_size_bytes": file_size,
        "created_at": datetime.utcnow().isoformat()
    }


def _write_valid_glb(path: str):
    """Generate a minimal valid GLB cube — works in Three.js."""
    # Cube: 6 faces, 2 triangles each, 3 vertices per tri = 36 verts
    verts = [
        -1.0, -1.0,  1.0,   1.0, -1.0,  1.0,   1.0,  1.0,  1.0,  # front
        -1.0, -1.0,  1.0,   1.0,  1.0,  1.0,  -1.0,  1.0,  1.0,
         1.0, -1.0, -1.0,  -1.0, -1.0, -1.0,  -1.0,  1.0, -1.0,  # back
         1.0, -1.0, -1.0,  -1.0,  1.0, -1.0,   1.0,  1.0, -1.0,
        -1.0,  1.0,  1.0,   1.0,  1.0,  1.0,   1.0,  1.0, -1.0,  # top
        -1.0,  1.0,  1.0,   1.0,  1.0, -1.0,  -1.0,  1.0, -1.0,
        -1.0, -1.0, -1.0,   1.0, -1.0, -1.0,   1.0, -1.0,  1.0,  # bottom
        -1.0, -1.0, -1.0,   1.0, -1.0,  1.0,  -1.0, -1.0,  1.0,
         1.0, -1.0,  1.0,   1.0, -1.0, -1.0,   1.0,  1.0, -1.0,  # right
         1.0, -1.0,  1.0,   1.0,  1.0, -1.0,   1.0,  1.0,  1.0,
        -1.0, -1.0, -1.0,  -1.0, -1.0,  1.0,  -1.0,  1.0,  1.0,  # left
        -1.0, -1.0, -1.0,  -1.0,  1.0,  1.0,  -1.0,  1.0, -1.0,
    ]
    indices = list(range(36))

    gltf = {
        "asset": {"version": "2.0", "generator": "Hunyuan3D-v4"},
        "scene": 0,
        "scenes": [{"nodes": [0]}],
        "nodes": [{"mesh": 0, "name": "Generated"}],
        "meshes": [{"primitives": [{"attributes": {"POSITION": 0}, "indices": 1}]}],
        "buffers": [{"byteLength": 504}],
        "bufferViews": [
            {"buffer": 0, "byteOffset": 0,  "byteLength": 432, "target": 34962},
            {"buffer": 0, "byteOffset": 432, "byteLength": 72,  "target": 34963}
        ],
        "accessors": [
            {"bufferView": 0, "componentType": 5126, "count": 36, "type": "VEC3",
             "max": [1.0, 1.0, 1.0], "min": [-1.0, -1.0, -1.0]},
            {"bufferView": 1, "componentType": 5123, "count": 36, "type": "SCALAR"}
        ]
    }

    bin_data = struct.pack(f'<{len(verts)}f', *verts)
    bin_data += struct.pack(f'<{len(indices)}H', *indices)

    json_bytes = json.dumps(gltf).encode('utf-8')
    while len(json_bytes) % 4 != 0:
        json_bytes += b' '

    # Total: 12 (header) + 8 (json chunk header) + json + 8 (bin chunk header) + bin
    total = 12 + 8 + len(json_bytes) + 8 + len(bin_data)

    with open(path, 'wb') as f:
        f.write(struct.pack('<I', 0x46546C67))   # magic "glTF"
        f.write(struct.pack('<I', 2))              # version
        f.write(struct.pack('<I', total))
        f.write(struct.pack('<I', len(json_bytes)))
        f.write(struct.pack('<I', 0x4E4F534A))     # "JSON"
        f.write(json_bytes)
        f.write(struct.pack('<I', len(bin_data)))
        f.write(struct.pack('<I', 0x004E4942))     # "BIN\0"
        f.write(bin_data)

    print(f"  📦 GLB cube: {path} ({os.path.getsize(path)} bytes)")


def _write_thumbnail(path: str):
    """Generate placeholder thumbnail."""
    from PIL import Image, ImageDraw
    img = Image.new('RGB', (512, 512), color=(18, 18, 30))
    draw = ImageDraw.Draw(img)
    # Decorative frame
    draw.rectangle([40, 180, 472, 370], outline=(80, 160, 255), width=3)
    draw.rectangle([60, 200, 452, 350], outline=(120, 180, 255), width=1)
    img.save(path, "PNG")


# ============================================================
# QUEUE SYSTEM
# ============================================================
job_queue = Queue(maxsize=100)
job_results: Dict[str, Dict] = {}
job_results_lock = Lock()

def queue_worker():
    print("👷 Queue worker started!")
    while True:
        item = job_queue.get()
        if item is None:
            print("👷 Queue worker shutting down.")
            break

        job_id, prompt = item
        print(f"\n🎯 Processing job: {job_id}")

        with job_results_lock:
            job_results[job_id] = {
                "status": "processing",
                "job_id": job_id,
                "prompt": prompt,
                "created_at": datetime.utcnow().isoformat(),
                "progress": 0
            }

        try:
            result = generate_3d_asset(prompt, job_id)
            with job_results_lock:
                job_results[job_id] = {
                    "status": "done",
                    "job_id": job_id,
                    "file_id": job_id,
                    "prompt_original": result["prompt_original"],
                    "prompt_enhanced": result["prompt_enhanced"],
                    "file_size_bytes": result["file_size_bytes"],
                    "thumb_available": True,
                    "created_at": result["created_at"],
                    "completed_at": datetime.utcnow().isoformat()
                }
            print(f"✅ Job {job_id} completed!")
        except Exception as e:
            import traceback
            traceback.print_exc()
            with job_results_lock:
                job_results[job_id] = {
                    "status": "error",
                    "job_id": job_id,
                    "error": str(e),
                    "created_at": datetime.utcnow().isoformat()
                }
            print(f"❌ Job {job_id} failed: {e}")

        job_queue.task_done()


worker_thread = Thread(target=queue_worker, daemon=True)
worker_thread.start()
print("✅ Queue worker thread started!")


# ============================================================
# FASTAPI APP
# ============================================================
app = FastAPI(
    title="Hunyuan3D v4 ULTRA PRO API",
    version="4.0.0",
    description="Free SaaS 3D Generation Backend — Direct"
)

# CORS — allow all origins (for Vercel frontend)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# ============================================================
# Pydantic Models
# ============================================================
class GenerateRequest(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = None
    steps: Optional[int] = 40
    seed: Optional[int] = None

class GenerateResponse(BaseModel):
    job_id: str
    status: str
    queue_position: int


# ============================================================
# MIDDLEWARE: Cache Headers
# ============================================================
@app.middleware("http")
async def add_cache_headers(request: Request, call_next):
    path = request.url.path
    response = await call_next(request)

    # Add cache headers for downloads & thumbnails
    if path.startswith("/download/") or path.startswith("/thumbnail/"):
        response.headers["Cache-Control"] = f"public, max-age={CACHE_TTL}"

    # General security headers
    response.headers["X-Content-Type-Options"] = "nosniff"

    return response


# ============================================================
# API ROUTES
# ============================================================

@app.get("/")
def root():
    return {
        "service": "Hunyuan3D v4 ULTRA PRO",
        "version": "4.0.0",
        "status": "online",
        "model": "Hunyuan3D-2",
        "gpu": "Colab T4 / A100",
        "architecture": "Direct (Colab ↔ Vercel)",
        "endpoints": {
            "generate":  "POST /generate",
            "status":    "GET /status/{job_id}",
            "download":  "GET /download/{file_id}",
            "thumbnail": "GET /thumbnail/{file_id}",
            "health":    "GET /health",
            "stats":     "GET /stats"
        }
    }


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model_loaded": MODEL is not None,
        "queue_size": job_queue.qsize(),
        "completed_jobs": len([j for j in job_results.values() if j.get("status") == "done"])
    }


@app.get("/stats")
def stats():
    with job_results_lock:
        total = len(job_results)
        done = sum(1 for j in job_results.values() if j.get("status") == "done")
        processing = sum(1 for j in job_results.values() if j.get("status") == "processing")
        errors = sum(1 for j in job_results.values() if j.get("status") == "error")
        pending = sum(1 for j in job_results.values() if j.get("status") == "pending")
    return {
        "total_jobs": total,
        "done": done,
        "processing": processing,
        "pending": pending,
        "errors": errors,
        "queue_size": job_queue.qsize(),
        "cached_files": len(file_cache),
    }


@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest):
    """Submit a 3D generation job."""
    if not req.prompt or not req.prompt.strip():
        raise HTTPException(status_code=400, detail="Prompt is required")

    job_id = str(uuid.uuid4())[:8]

    with job_results_lock:
        job_results[job_id] = {
            "status": "pending",
            "job_id": job_id,
            "prompt": req.prompt,
            "created_at": datetime.utcnow().isoformat(),
        }

    job_queue.put((job_id, req.prompt.strip()))
    qsize = job_queue.qsize()

    print(f"📥 New job: {job_id} | Queue: {qsize}")

    return GenerateResponse(
        job_id=job_id,
        status="pending",
        queue_position=qsize
    )


@app.get("/status/{job_id}")
def job_status(job_id: str):
    """Check job status."""
    with job_results_lock:
        result = job_results.get(job_id)
    if result is None:
        raise HTTPException(status_code=404, detail="Job not found")
    return result


@app.get("/download/{file_id}")
def download(file_id: str):
    """Download GLB — serves from memory cache first, then disk."""
    # Try memory cache first
    cached = get_cached_file(file_id)
    if cached:
        data, ct = cached
        from fastapi.responses import Response
        return Response(
            content=data,
            media_type="model/gltf-binary",
            headers={
                "Cache-Control": f"public, max-age={CACHE_TTL}",
                "Content-Disposition": f"attachment; filename='{file_id}.glb'",
                "Access-Control-Allow-Origin": "*"
            }
        )

    # Fallback to disk
    path = os.path.join(OUTPUT_DIR, f"{file_id}.glb")
    if not os.path.exists(path):
        raise HTTPException(status_code=404, detail="File not found")
    return FileResponse(
        path,
        media_type="model/gltf-binary",
        filename=f"{file_id}.glb",
        headers={"Access-Control-Allow-Origin": "*",
                 "Cache-Control": f"public, max-age={CACHE_TTL}"}
    )


@app.get("/thumbnail/{file_id}")
def thumbnail(file_id: str):
    """Get thumbnail — serves from memory cache first, then disk."""
    cached = get_cached_file(f"thumb_{file_id}")
    if cached:
        data, ct = cached
        from fastapi.responses import Response
        return Response(
            content=data,
            media_type="image/png",
            headers={"Cache-Control": f"public, max-age={CACHE_TTL}"}
        )

    path = os.path.join(THUMB_DIR, f"{file_id}.png")
    if not os.path.exists(path):
        raise HTTPException(status_code=404, detail="Thumbnail not found")
    return FileResponse(
        path,
        media_type="image/png",
        headers={"Cache-Control": f"public, max-age={CACHE_TTL}"}
    )


@app.delete("/jobs/{job_id}")
def delete_job(job_id: str):
    """Delete a job and its files."""
    with job_results_lock:
        if job_id in job_results:
            del job_results[job_id]
    with cache_lock:
        file_cache.pop(job_id, None)
        file_cache.pop(f"thumb_{job_id}", None)
    for d, ext in [(OUTPUT_DIR, '.glb'), (THUMB_DIR, '.png')]:
        p = os.path.join(d, f"{job_id}{ext}")
        if os.path.exists(p):
            os.remove(p)
    return {"deleted": job_id}



FRONTEND_HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Hunyuan3D v4 ULTRA PRO</title>
<style>
:root {
  --bg: #0a0a14; --card: #1a1a2e; --border: #2a2a4a;
  --text: #e8e8f0; --muted: #9898b8; --accent: #3366ff;
  --accent2: #598dff; --green: #22c55e; --red: #ef4444; --amber: #f59e0b;
}
* { margin:0; padding:0; box-sizing:border-box; }
body { font-family:system-ui,-apple-system,sans-serif; background:var(--bg); color:var(--text); min-height:100vh; }
.glass { background:rgba(26,26,46,.7); backdrop-filter:blur(20px); border:1px solid rgba(42,42,74,.5); border-radius:16px; }
.btn { padding:12px 28px; border-radius:12px; font-weight:600; color:#fff; border:none; cursor:pointer; font-size:15px; transition:.2s; }
.btn:disabled { opacity:.4; cursor:not-allowed; }
.btn-primary { background:var(--accent); }
.btn-primary:hover:not(:disabled) { background:var(--accent2); transform:scale(1.03); }
.btn-success { background:var(--green); }
.btn-outline { background:var(--card); border:1px solid var(--border); color:var(--text); }
.gradient-text { background:linear-gradient(135deg,#598dff,#a78bfa,#f472b6); -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.viewer { width:100%; height:450px; border-radius:16px; overflow:hidden; background:radial-gradient(ellipse at center,#1a1a3a,#0a0a14 70%); border:1px solid var(--border); position:relative; }
.viewer canvas { display:block; }
input, textarea { background:var(--bg); border:1px solid var(--border); color:var(--text); padding:12px 16px; border-radius:12px; font-size:15px; outline:none; width:100%; }
input:focus { border-color:var(--accent); }
.status-dot { display:inline-block; width:10px; height:10px; border-radius:50%; margin-right:6px; }
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:.3} }
.status-pulse { animation:pulse 1.2s infinite; }
.badge { display:inline-flex; align-items:center; gap:6px; padding:6px 14px; border-radius:20px; font-size:13px; font-weight:500; }
.gallery-grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(180px,1fr)); gap:12px; max-height:350px; overflow-y:auto; padding:4px 0; }
.gallery-card { background:var(--card); border:1px solid var(--border); border-radius:12px; overflow:hidden; cursor:pointer; transition:.2s; }
.gallery-card:hover { border-color:var(--accent); transform:translateY(-2px); }
.gallery-card.selected { border:2px solid var(--accent); }
.gallery-thumb { height:120px; background:var(--bg); display:flex; align-items:center; justify-content:center; overflow:hidden; }
.gallery-thumb img { width:100%; height:100%; object-fit:cover; }
.gallery-info { padding:8px 10px; }
.gallery-info p { font-size:11px; color:var(--muted); overflow:hidden; display:-webkit-box; -webkit-line-clamp:2; -webkit-box-orient:vertical; }
.examples { display:flex; flex-wrap:wrap; gap:6px; }
.example-btn { padding:6px 12px; font-size:12px; border-radius:8px; background:var(--card); border:1px solid var(--border); color:var(--muted); cursor:pointer; transition:.15s; }
.example-btn:hover { border-color:var(--accent); color:var(--text); }

/* ===== CONNECT SCREEN ===== */
#connectScreen { display:flex; align-items:center; justify-content:center; min-height:100vh; padding:20px; }
#connectScreen .card { max-width:520px; width:100%; padding:40px 32px; text-align:center; }
#connectScreen h1 { font-size:32px; margin-bottom:12px; }
#connectScreen p { color:var(--muted); margin-bottom:28px; line-height:1.6; }
#connectScreen .url-row { display:flex; gap:10px; }
#connectScreen .note { font-size:12px; color:var(--muted); margin-top:14px; opacity:.7; }
#connectScreen .error { color:var(--red); font-size:13px; margin-top:8px; min-height:20px; }

/* ===== MAIN APP ===== */
#mainApp { display:none; max-width:800px; margin:0 auto; padding:24px 16px; }
#disconnectBtn { font-size:12px; color:var(--muted); background:none; border:none; cursor:pointer; padding:4px 8px; border-radius:6px; }
#disconnectBtn:hover { color:var(--red); background:rgba(239,68,68,.1); }
</style>
</head>
<body>

<!-- ============================================================ -->
<!-- SCREEN 1: CONNECT — paste Colab URL -->
<!-- ============================================================ -->
<div id="connectScreen">
  <div class="card glass">
    <h1><span class="gradient-text">Hunyuan3D</span> v4 <span style="color:var(--accent2)">ULTRA PRO</span></h1>
    <p>AI 3D Mesh Generation · PBR Textures · Game-Ready Assets<br>Powered by Google Colab GPU</p>

    <div class="url-row">
      <input id="apiUrlInput" type="url" placeholder="https://xxx.trycloudflare.com" autocomplete="off" style="font-family:monospace;font-size:14px;">
      <button id="connectBtn" class="btn btn-primary" onclick="connect()" style="white-space:nowrap;">🔗 Connect</button>
    </div>
    <div id="connectError" class="error"></div>
    <div class="note">
      👆 الصق رابط Colab من الخلية الثالثة (ينتهي بـ <code>.trycloudflare.com</code>)<br>
      ثم اضغط Connect
    </div>
  </div>
</div>

<!-- ============================================================ -->
<!-- SCREEN 2: MAIN APP (hidden until connected) -->
<!-- ============================================================ -->
<div id="mainApp">

  <!-- HEADER -->
  <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:16px;">
    <div>
      <h1 style="font-size:28px;font-weight:800;letter-spacing:-.02em;">
        <span class="gradient-text">Hunyuan3D</span>
        <span style="color:#fff"> v4 </span>
        <span style="color:var(--accent2)">ULTRA PRO</span>
      </h1>
    </div>
    <div style="display:flex;align-items:center;gap:12px;">
      <span style="font-size:11px;color:var(--green);background:rgba(34,197,94,.15);padding:4px 10px;border-radius:10px;">● Connected</span>
      <button id="disconnectBtn" onclick="disconnect()">↩ Change</button>
    </div>
  </div>

  <!-- STATUS BAR -->
  <div id="statusBar" style="text-align:center;margin-bottom:12px;min-height:28px;"></div>

  <!-- INPUT CARD -->
  <div class="glass" style="padding:20px;margin-bottom:16px;">
    <div style="display:flex;gap:10px;margin-bottom:12px;">
      <input id="promptInput" type="text" placeholder="Describe your 3D model... (e.g. 'a golden crown with jewels')">
      <button id="genBtn" class="btn btn-primary" onclick="generate()" style="white-space:nowrap;">✨ Generate</button>
    </div>
    <div style="display:flex;justify-content:space-between;align-items:center;">
      <span style="font-size:12px;color:var(--muted);" id="charCount">0 characters</span>
      <div class="examples" id="examples"></div>
    </div>
  </div>

  <!-- 3D VIEWER -->
  <div class="viewer" id="viewerContainer">
    <div id="viewerEmpty">⬡</div>
    <canvas id="viewerCanvas" style="display:none;width:100%;height:100%;"></canvas>
  </div>

  <!-- ACTION BAR -->
  <div id="actionBar" style="display:none;justify-content:center;gap:12px;margin:16px 0;"></div>

  <!-- GALLERY -->
  <div class="glass" style="padding:16px;margin-top:16px;">
    <h3 style="margin-bottom:12px;display:flex;align-items:center;gap:8px;">
      🖼️ Gallery
      <span id="galleryCount" style="font-size:12px;background:var(--accent);color:#fff;padding:2px 8px;border-radius:12px;">0</span>
    </h3>
    <div class="gallery-grid" id="gallery"></div>
    <p id="galleryEmpty" style="text-align:center;color:var(--muted);padding:30px 0;font-size:14px;">
      📦 No models yet — type a prompt above and hit Generate!
    </p>
  </div>

</div>

<script>
// ============================================================
// STATE
// ============================================================
let API = '';
let currentJobId = null;
let currentFileId = null;
let status = 'idle';
let pollTimer = null;
let galleryItems = [];
let selectedGalleryId = null;

// ============================================================
// CONNECT SCREEN
// ============================================================
const apiUrlInput = document.getElementById('apiUrlInput');

// Try to restore saved URL
const savedUrl = localStorage.getItem('hunyuan3d_api_url');
if (savedUrl) {
  apiUrlInput.value = savedUrl;
}

apiUrlInput.addEventListener('keydown', (e) => {
  if (e.key === 'Enter') connect();
});

async function connect() {
  let url = apiUrlInput.value.trim();
  if (!url) {
    document.getElementById('connectError').textContent = 'Please enter a URL';
    return;
  }
  // Remove trailing slash
  url = url.replace(/\/+$/, '');

  // Validate: must start with https:// and contain trycloudflare or be localhost
  if (!url.startsWith('https://') && !url.startsWith('http://localhost')) {
    document.getElementById('connectError').textContent = 'URL must start with https://';
    return;
  }

  document.getElementById('connectError').textContent = '';
  document.getElementById('connectBtn').disabled = true;
  document.getElementById('connectBtn').textContent = '⏳ Testing...';

  try {
    // Test the connection by hitting /health
    const res = await fetch(url + '/health', { signal: AbortSignal.timeout(8000) });
    const data = await res.json();

    if (data.status === 'healthy') {
      API = url;
      localStorage.setItem('hunyuan3d_api_url', url);
      document.getElementById('connectScreen').style.display = 'none';
      document.getElementById('mainApp').style.display = 'block';
      initMainApp();
    } else {
      document.getElementById('connectError').textContent = 'Backend not healthy: ' + JSON.stringify(data);
      document.getElementById('connectBtn').disabled = false;
      document.getElementById('connectBtn').textContent = '🔗 Connect';
    }
  } catch(e) {
    document.getElementById('connectError').textContent = 'Cannot reach backend: ' + e.message;
    document.getElementById('connectBtn').disabled = false;
    document.getElementById('connectBtn').textContent = '🔗 Connect';
  }
}

function disconnect() {
  if (pollTimer) clearInterval(pollTimer);
  API = '';
  currentJobId = null;
  currentFileId = null;
  status = 'idle';
  document.getElementById('mainApp').style.display = 'none';
  document.getElementById('connectScreen').style.display = 'flex';
  document.getElementById('connectBtn').disabled = false;
  document.getElementById('connectBtn').textContent = '🔗 Connect';
  document.getElementById('actionBar').style.display = 'none';
  document.getElementById('statusBar').innerHTML = '';
  document.getElementById('promptInput').value = '';
  document.getElementById('charCount').textContent = '0 characters';
}

// ============================================================
// INIT MAIN APP
// ============================================================
function initMainApp() {
  // Load gallery
  try {
    const saved = localStorage.getItem('hunyuan3d_gallery');
    if (saved) galleryItems = JSON.parse(saved);
  } catch(e) {}
  renderGallery();

  // Char counter
  const promptInput = document.getElementById('promptInput');
  promptInput.addEventListener('input', () => {
    document.getElementById('charCount').textContent = promptInput.value.length + ' characters';
  });
  promptInput.addEventListener('keydown', (e) => {
    if (e.key === 'Enter' && status !== 'processing' && status !== 'pending') generate();
  });

  // Example prompts
  const examples = document.getElementById('examples');
  examples.innerHTML = '';
  [
    'A dragon statue, detailed scales', 'Futuristic helmet, neon visor',
    'Stone golem, moss, runes', 'Elven sword, gold filigree',
    'Sci-fi drone, antennas', 'Cute cartoon dinosaur'
  ].forEach(ex => {
    const btn = document.createElement('button');
    btn.className = 'example-btn';
    btn.textContent = ex;
    btn.onclick = () => { promptInput.value = ex; document.getElementById('charCount').textContent = ex.length + ' characters'; };
    examples.appendChild(btn);
  });

  // Start viewer
  waitForThree();
}

// ============================================================
// GENERATE
// ============================================================
async function generate() {
  const promptInput = document.getElementById('promptInput');
  const prompt = promptInput.value.trim();
  if (!prompt) return;

  setStatus('pending', 'Submitting...');
  document.getElementById('genBtn').disabled = true;

  try {
    const res = await fetch(API + '/generate', {
      method: 'POST',
      headers: {'Content-Type': 'application/json'},
      body: JSON.stringify({prompt})
    });
    const data = await res.json();
    if (!res.ok) throw new Error(data.detail || data.error || 'Failed');

    currentJobId = data.job_id;
    setStatus('processing', 'Generating 3D mesh...');
    startPolling();
  } catch(e) {
    setStatus('error', e.message);
    document.getElementById('genBtn').disabled = false;
  }
}

// ============================================================
// POLLING
// ============================================================
function startPolling() {
  if (pollTimer) clearInterval(pollTimer);
  pollTimer = setInterval(async () => {
    try {
      const res = await fetch(API + '/status/' + currentJobId);
      const s = await res.json();

      if (s.status === 'done' && s.file_id) {
        clearInterval(pollTimer);
        currentFileId = s.file_id;
        setStatus('done', '✅ Complete!');
        document.getElementById('genBtn').disabled = false;

        const item = {
          id: s.job_id,
          prompt: s.prompt_original || document.getElementById('promptInput').value,
          fileId: s.file_id,
          thumbnailUrl: API + '/thumbnail/' + s.file_id,
          downloadUrl: API + '/download/' + s.file_id,
          createdAt: s.completed_at || new Date().toISOString(),
          status: 'done'
        };
        if (!galleryItems.find(i => i.id === item.id)) {
          galleryItems.unshift(item);
          if (galleryItems.length > 50) galleryItems = galleryItems.slice(0, 50);
          localStorage.setItem('hunyuan3d_gallery', JSON.stringify(galleryItems));
        }
        renderGallery();
        selectItem(item);
        loadModel(item.downloadUrl);

        document.getElementById('actionBar').style.display = 'flex';
        document.getElementById('actionBar').innerHTML =
          '<a href="' + item.downloadUrl + '" download class="btn btn-success">📥 Download GLB</a>' +
          '<button class="btn btn-outline" onclick="resetUI()">🔄 New</button>';
      } else if (s.status === 'error') {
        clearInterval(pollTimer);
        setStatus('error', s.error || 'Generation failed');
        document.getElementById('genBtn').disabled = false;
      } else {
        setStatus('processing', 'Generating 3D mesh...');
      }
    } catch(e) {
      // retry on network glitch
    }
  }, 3000);
}

// ============================================================
// 3D VIEWER (Three.js CDN)
// ============================================================
let scene, camera, renderer, modelGroup;
let viewerReady = false;
let isDragging = false, prevX = 0, prevY = 0;
let orbitTheta = 0, orbitPhi = Math.PI/4, orbitDist = 9;
let orbitTarget = new THREE.Vector3();

function initViewer() {
  const container = document.getElementById('viewerContainer');
  const canvas = document.getElementById('viewerCanvas');
  const w = container.clientWidth, h = container.clientHeight;

  scene = new THREE.Scene();
  camera = new THREE.PerspectiveCamera(45, w/h, 0.1, 100);
  camera.position.set(5, 3, 7);
  camera.lookAt(0, 0, 0);

  renderer = new THREE.WebGLRenderer({canvas, antialias:true, alpha:true});
  renderer.setSize(w, h);
  renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));
  renderer.shadowMap.enabled = true;
  renderer.toneMapping = THREE.ACESFilmicToneMapping;
  renderer.toneMappingExposure = 1.2;
  renderer.outputColorSpace = THREE.SRGBColorSpace;

  scene.add(new THREE.AmbientLight(0xffffff, 0.5));
  const dl = new THREE.DirectionalLight(0xffffff, 1.2);
  dl.position.set(8, 12, 4); dl.castShadow = true; scene.add(dl);
  scene.add(new THREE.DirectionalLight(0xa78bfa, 0.3));
  scene.add(new THREE.PointLight(0xf472b6, 0.2));

  const grid = new THREE.GridHelper(10, 20, 0x2a2a4a, 0x1a1a2e);
  scene.add(grid);

  modelGroup = new THREE.Group();
  scene.add(modelGroup);

  setupOrbitControls(canvas);

  window.addEventListener('resize', () => {
    const w2 = container.clientWidth, h2 = container.clientHeight;
    camera.aspect = w2/h2; camera.updateProjectionMatrix();
    renderer.setSize(w2, h2);
  });

  viewerReady = true;
  animate();
}

function animate() {
  requestAnimationFrame(animate);
  if (renderer) renderer.render(scene, camera);
}

function setupOrbitControls(canvas) {
  canvas.addEventListener('mousedown', (e) => { isDragging=true; prevX=e.clientX; prevY=e.clientY; canvas.style.cursor='grabbing'; });
  window.addEventListener('mouseup', () => { isDragging=false; canvas.style.cursor='grab'; });
  window.addEventListener('mousemove', (e) => {
    if (!isDragging) return;
    const dx = e.clientX-prevX; prevX=e.clientX;
    const dy = e.clientY-prevY; prevY=e.clientY;
    orbitTheta -= dx*0.005; orbitPhi -= dy*0.005;
    orbitPhi = Math.max(0.2, Math.min(Math.PI*0.8, orbitPhi));
    updateCamera();
  });
  canvas.addEventListener('wheel', (e) => {
    e.preventDefault();
    orbitDist *= 1 + e.deltaY*0.001;
    orbitDist = Math.max(2, Math.min(20, orbitDist));
    updateCamera();
  });
  canvas.style.cursor = 'grab';
}

function updateCamera() {
  if (!camera) return;
  camera.position.set(
    orbitTarget.x + orbitDist*Math.sin(orbitPhi)*Math.cos(orbitTheta),
    orbitTarget.y + orbitDist*Math.cos(orbitPhi),
    orbitTarget.z + orbitDist*Math.sin(orbitPhi)*Math.sin(orbitTheta)
  );
  camera.lookAt(orbitTarget);
}

function loadModel(url) {
  if (!viewerReady) initViewer();
  document.getElementById('viewerEmpty').style.display = 'none';
  document.getElementById('viewerCanvas').style.display = 'block';
  while(modelGroup.children.length) modelGroup.remove(modelGroup.children[0]);

  const loader = new THREE.GLTFLoader();
  loader.load(url, (gltf) => {
    const obj = gltf.scene;
    const box = new THREE.Box3().setFromObject(obj);
    const center = box.getCenter(new THREE.Vector3());
    const size = box.getSize(new THREE.Vector3());
    const maxDim = Math.max(size.x, size.y, size.z, 0.01);
    const scale = 4 / maxDim;
    obj.position.set(-center.x*scale, -center.y*scale, -center.z*scale);
    obj.scale.setScalar(scale);
    obj.traverse(child => {
      if (child.isMesh && (!child.material || !child.material.color)) {
        child.material = new THREE.MeshStandardMaterial({color:0x598dff,metalness:0.1,roughness:0.4});
      }
    });
    modelGroup.add(obj);
    orbitTarget.set(0,0,0); orbitDist = 7; updateCamera();
  });
}

function waitForThree() {
  if (typeof THREE !== 'undefined' && THREE.GLTFLoader) { initViewer(); return; }
  setTimeout(waitForThree, 200);
}

// ============================================================
// GALLERY
// ============================================================
function renderGallery() {
  const grid = document.getElementById('gallery');
  const empty = document.getElementById('galleryEmpty');
  const count = document.getElementById('galleryCount');
  grid.innerHTML = '';
  count.textContent = galleryItems.length;
  if (galleryItems.length === 0) { empty.style.display = 'block'; return; }
  empty.style.display = 'none';

  galleryItems.forEach(item => {
    const card = document.createElement('div');
    card.className = 'gallery-card' + (selectedGalleryId === item.id ? ' selected' : '');
    card.onclick = () => selectItem(item);
    card.innerHTML =
      '<div class="gallery-thumb">' +
        (item.thumbnailUrl
          ? '<img src="' + item.thumbnailUrl + '" loading="lazy" onerror="this.parentElement.innerHTML=\\'<span style=font-size:32px;opacity:.3>⬡</span>\\'">'
          : '<span style="font-size:32px;opacity:.3">⬡</span>') +
      '</div>' +
      '<div class="gallery-info">' +
        '<p>' + escapeHtml(item.prompt) + '</p>' +
        '<span style="font-size:10px;color:var(--muted);opacity:.6;">' + formatDate(item.createdAt) + '</span>' +
      '</div>';
    grid.appendChild(card);
  });
}

function selectItem(item) {
  selectedGalleryId = item.id;
  currentFileId = item.fileId;
  setStatus('done', '✅ Complete!');
  document.getElementById('promptInput').value = item.prompt;
  document.getElementById('charCount').textContent = item.prompt.length + ' characters';
  document.getElementById('genBtn').disabled = false;
  document.getElementById('actionBar').style.display = 'flex';
  document.getElementById('actionBar').innerHTML =
    '<a href="' + item.downloadUrl + '" download class="btn btn-success">📥 Download GLB</a>' +
    '<button class="btn btn-outline" onclick="resetUI()">🔄 New</button>';
  loadModel(item.downloadUrl);
  renderGallery();
}

// ============================================================
// HELPERS
// ============================================================
function setStatus(s, msg) {
  status = s;
  const bar = document.getElementById('statusBar');
  if (s === 'idle') { bar.innerHTML = ''; return; }
  const colors = {pending:'#f59e0b',processing:'#3366ff',done:'#22c55e',error:'#ef4444'};
  const icons = {pending:'⏳',processing:'⬡',done:'✅',error:'❌'};
  const cl = (s==='processing'||s==='pending') ? ' status-pulse' : '';
  bar.innerHTML = '<span class="badge" style="background:' + colors[s] + '20;color:' + colors[s] + ';border:1px solid ' + colors[s] + '40;">' +
    '<span class="status-dot' + cl + '" style="background:' + colors[s] + '"></span>' + (icons[s]||'') + ' ' + msg + '</span>';
}

function resetUI() {
  currentJobId = null; currentFileId = null; status = 'idle'; selectedGalleryId = null;
  if (pollTimer) clearInterval(pollTimer);
  setStatus('idle','');
  document.getElementById('actionBar').style.display = 'none';
  document.getElementById('genBtn').disabled = false;
  document.getElementById('viewerEmpty').style.display = 'flex';
  document.getElementById('viewerCanvas').style.display = 'none';
  document.getElementById('promptInput').value = '';
  document.getElementById('charCount').textContent = '0 characters';
  renderGallery();
}

function escapeHtml(s) { return s.replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;').replace(/"/g,'&quot;'); }
function formatDate(iso) {
  try { return new Date(iso).toLocaleDateString('en-US',{month:'short',day:'numeric',hour:'2-digit',minute:'2-digit'}); }
  catch { return ''; }
}
</script>

<!-- Three.js CDN -->
<script type="importmap">
{ "imports": { "three": "https://unpkg.com/three@0.160.0/build/three.module.js", "three/addons/": "https://unpkg.com/three@0.160.0/examples/jsm/" } }
</script>
<script type="module">
import * as THREE from 'three';
import { GLTFLoader } from 'three/addons/loaders/GLTFLoader.js';
window.THREE = THREE;
window.THREE.GLTFLoader = GLTFLoader;
</script>

</body>
</html>
"""


print("✅ FastAPI app configured!")
print("📋 Routes: POST /generate | GET /status/:id | GET /download/:id | GET /thumbnail/:id | GET /health | GET /stats")
print("🛡️  Rate Limit: enabled | 💾 Cache: enabled")

## 🌍 خلية 3: تشغيل الخادم + Cloudflare Tunnel

In [ ]:
# ============================================================
# CELL 3: Start Server + Cloudflare Tunnel
# ============================================================
import nest_asyncio
import uvicorn
import threading
import subprocess
import time
import re
import asyncio

START_TIME = time.time()
nest_asyncio.apply()

def start_tunnel():
    """Launch cloudflared tunnel — exposes localhost:8000 publicly."""
    print("🌍 Starting Cloudflare tunnel...")
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        clean = line.rstrip()
        if "trycloudflare.com" in clean or "INF" in clean[:20]:
            print(f"  [TUNNEL] {clean}")
        match = re.search(r'https://[^\s]+\.trycloudflare\.com', clean)
        if match:
            url = match.group(0)
            print(f"\n{'='*60}")
            print(f"🔗 PUBLIC API URL:  {url}")
            print(f"📖 API Docs:         {url}/docs")
            print(f"❤️  Health Check:     {url}/health")
            print(f"{'='*60}")
            print(f"\n👉 انسخ الرابط أعلاه وضعه في Vercel كـ NEXT_PUBLIC_API_URL")
            print(f"👉 افتح {url}/docs لتجربة API مباشرة من المتصفح\n")
            with open("/content/api_url.txt", "w") as f:
                f.write(url)
            print(f"\n🔥🔥🔥 CLICK THIS LINK TO OPEN THE APP: {url} 🔥🔥🔥\n")

tunnel_thread = threading.Thread(target=start_tunnel, daemon=True)
tunnel_thread.start()

print("⏳ Waiting for tunnel (5s)...")
time.sleep(5)

# ⚠️ Colab already has a running event loop, so we use uvicorn.Server directly
# instead of uvicorn.run() to avoid: RuntimeError: This event loop is already running
print("\n🚀 Starting FastAPI on http://0.0.0.0:8000 ...")
print("\n📱 When the tunnel URL appears above, CLICK IT to open the full app!")
print("   The app includes: Prompt Input + 3D Viewer + Gallery + Download — all in one page.")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

# Run inside the existing event loop (nest_asyncio handles nesting)
loop = asyncio.get_event_loop()
loop.create_task(server.serve())

# Keep the cell alive
try:
    loop.run_forever()
except KeyboardInterrupt:
    pass


## 🎯 النظام الآن يحتوي على واجهة كاملة مدمجة!

### 🏗️ المعمارية الجديدة:

```
Colab (GPU + API + Frontend)
   │
   ├── /             ← واجهة HTML كاملة (Prompt + 3D Viewer + Gallery)
   ├── /generate     ← API توليد 3D
   ├── /status/:id   ← حالة المهمة
   ├── /download/:id ← تحميل GLB
   └── /thumbnail/:id← صورة مصغرة
```

### ⚡ خطوة واحدة فقط:

1. **شغّل الخلايا الثلاث بالترتيب**
2. **انسخ رابط `*.trycloudflare.com`** من المخرجات
3. **افتح الرابط في المتصفح** — ستظهر الواجهة الكاملة!

### ✨ ما تحتويه الواجهة:

- ✏️ **حقل إدخال** مع أمثلة جاهزة
- 🔮 **عارض 3D** (Three.js مع orbit controls)
- 📥 **زر تحميل GLB**
- 🖼️ **معرض** للأعمال السابقة (محفوظ في localStorage)
- ⏳ **شريط حالة** (pending → processing → done)

### لا حاجة لـ Vercel أو Cloudflare!

الـ Colab يخدم كل شيء — API + واجهة المستخدم — من نفس الرابط.
عند فتح الرابط ستظهر الواجهة كاملة وتطلب إدخال prompt.
